# 🏗️ Python Data Structures & Classes - Complete Reference

A comprehensive guide to dataclasses, TypedDict, NamedTuple, regular classes, and when to use each.

---

## Table of Contents

1. [Overview: Choosing the Right Structure](#1-overview-choosing-the-right-structure)
2. [Regular Classes](#2-regular-classes)
3. [Dataclasses](#3-dataclasses)
4. [Frozen Dataclasses](#4-frozen-dataclasses)
5. [TypedDict](#5-typeddict)
6. [NamedTuple](#6-namedtuple)
7. [Enum](#7-enum)
8. [Attrs (Third-Party)](#8-attrs-third-party)
9. [Pydantic (Third-Party)](#9-pydantic-third-party)
10. [Comparison & Best Practices](#10-comparison--best-practices)
11. [Real-World Patterns](#11-real-world-patterns)

---

## 1. Overview: Choosing the Right Structure

Python offers multiple ways to create structured data. Here's a quick decision guide:

| Need | Best Choice |
|------|-------------|
| Simple data container | `dataclass` |
| Immutable data | `@dataclass(frozen=True)` or `NamedTuple` |
| Dict-like with type hints | `TypedDict` |
| JSON/API responses | `TypedDict` |
| Behavior + data | Regular class |
| Set of constants | `Enum` |
| Validation at runtime | `Pydantic` |
| Hashable, immutable | `NamedTuple` or frozen dataclass |

---

## 2. Regular Classes

Full control, but requires more boilerplate.

In [ ]:
# =============================================================================
# BASIC CLASS - Lots of boilerplate!
# =============================================================================

class Person:
    """A person with name and age."""
    
    def __init__(self, name: str, age: int) -> None:
        self.name = name
        self.age = age
    
    def __repr__(self) -> str:
        return f"Person(name={self.name!r}, age={self.age!r})"
    
    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Person):
            return NotImplemented
        return self.name == other.name and self.age == other.age
    
    def __hash__(self) -> int:
        return hash((self.name, self.age))


# Usage
person = Person("Manuel", 30)
print(person)  # Person(name='Manuel', age=30)

# Comparison works
person2 = Person("Manuel", 30)
print(person == person2)  # True

In [ ]:
# =============================================================================
# CLASS WITH METHODS AND PROPERTIES
# =============================================================================

class BankAccount:
    """Bank account with behavior."""
    
    def __init__(self, owner: str, balance: float = 0.0) -> None:
        self.owner = owner
        self._balance = balance  # Private by convention
        self._transactions: list[tuple[str, float]] = []
    
    @property
    def balance(self) -> float:
        """Read-only balance."""
        return self._balance
    
    def deposit(self, amount: float) -> None:
        """Deposit money."""
        if amount <= 0:
            raise ValueError("Amount must be positive")
        self._balance += amount
        self._transactions.append(("deposit", amount))
    
    def withdraw(self, amount: float) -> None:
        """Withdraw money."""
        if amount <= 0:
            raise ValueError("Amount must be positive")
        if amount > self._balance:
            raise ValueError("Insufficient funds")
        self._balance -= amount     
        self._transactions.append(("withdraw", amount))
    
    def __repr__(self) -> str:
        return f"BankAccount(owner={self.owner!r}, balance={self._balance})"


# Usage
account = BankAccount("Manuel", 100.0)
account.deposit(50.0)
account.withdraw(30.0)
print(account)  # BankAccount(owner='Manuel', balance=120.0)
print(f"Balance: ${account.balance}")  # Balance: $120.0

In [ ]:
# =============================================================================
# CLASS WITH __slots__ (Memory Optimization)
# =============================================================================

class Point:
    """
    Memory-efficient point using __slots__.
    
    __slots__ prevents dynamic attribute creation
    and reduces memory usage.
    """
    __slots__ = ('x', 'y')  # Only these attributes allowed
    
    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y
    
    def distance_from_origin(self) -> float:
        return (self.x ** 2 + self.y ** 2) ** 0.5
    
    def __repr__(self) -> str:
        return f"Point({self.x}, {self.y})"


point = Point(3.0, 4.0)
print(point)  # Point(3.0, 4.0)
print(f"Distance: {point.distance_from_origin()}")  # Distance: 5.0

# Cannot add new attributes
try:
    point.z = 5.0  # AttributeError!
except AttributeError as e:
    print(f"Error: {e}")

---

## 3. Dataclasses

Auto-generated `__init__`, `__repr__`, `__eq__` and more!

In [ ]:
# =============================================================================
# BASIC DATACLASS
# =============================================================================

from dataclasses import dataclass

@dataclass
class Person:
    """A person - much simpler than regular class!"""
    name: str
    age: int

# That's it! __init__, __repr__, __eq__ auto-generated!

person = Person("Manuel", 30)
print(person)  # Person(name='Manuel', age=30)

person2 = Person("Manuel", 30)
print(person == person2)  # True (auto-generated __eq__)

In [ ]:
# =============================================================================
# DATACLASS WITH DEFAULT VALUES
# =============================================================================

from dataclasses import dataclass
from typing import Final

@dataclass
class Config:
    """Application configuration."""
    
    # Required fields (no default) must come first
    app_name: str
    api_key: str
    
    # Optional fields with defaults
    debug: bool = False
    timeout: int = 30
    retries: int = 3
    max_connections: int = 100


# Use with defaults
config = Config("MyApp", "secret-key")
print(config)
# Config(app_name='MyApp', api_key='secret-key', debug=False, timeout=30, retries=3, max_connections=100)

# Override defaults
config2 = Config("MyApp", "secret-key", debug=True, timeout=60)
print(config2)

In [ ]:
# =============================================================================
# DATACLASS WITH MUTABLE DEFAULT VALUES
# =============================================================================

from dataclasses import dataclass, field

# ❌ WRONG: Mutable default shared across instances!
# @dataclass
# class WrongTodo:
#     title: str
#     tags: list[str] = []  # This is shared! Bug!


# ✅ CORRECT: Use field(default_factory=...)
@dataclass
class Todo:
    """Todo item with proper mutable default."""
    title: str
    done: bool = False
    tags: list[str] = field(default_factory=list)  # New list per instance
    metadata: dict = field(default_factory=dict)   # New dict per instance
# With a mutable default like tags: list = [], Python creates one list object at
# class definition time, and every instance gets a reference to that same list in memory.
# field(default_factory=list) tells Python "don't create the list now-call list()
# fresh for each new instance."
# Why Python doesn't handle this automatically: It's a deliberate design chouce inherited
# from regular functions. Default arguments in Python are evaluated once at definition time,
# not at call time. Same for dataclasses.

# The rule is simple: If the default is mutable (list, dict, set) use field(default_factory=...).
# If it's inmmutable(str, int, float, tupe, bool, None), a plain '=' is safe.

# Each instance gets its own list
todo1 = Todo("Buy groceries")
todo2 = Todo("Write code")

todo1.tags.append("shopping")
print(f"Todo1 tags: {todo1.tags}")  # ['shopping']
print(f"Todo2 tags: {todo2.tags}")  # [] (not affected!)

In [ ]:
# =============================================================================
# DATACLASS field() OPTIONS
# =============================================================================

from dataclasses import dataclass, field

@dataclass
class User:
    """User with various field options."""
    
    # Normal field
    name: str
    
    # Field with default
    active: bool = True
    
    # Field excluded from __repr__
    password: str = field(repr=False)
    
    # Field excluded from comparison
    login_count: int = field(default=0, compare=False)
    
    # Field excluded from __init__ (must have default)
    _internal_id: int = field(default=0, init=False, repr=False)
    
    # Field with factory
    roles: list[str] = field(default_factory=list)


user = User("Manuel", password="secret123")
print(user)
# User(name='Manuel', active=True, login_count=0, roles=[])
# Note: password not shown in repr!

# =============================================================================
# =============================================================================
parents: list[Person | None] = field(default_factory=lambda: [None, None])


# lambda with a parameter — you've used this pattern
sorted(names, key=lambda name: name.lower())
#                       ^^^^
#                  "name" is the input

# The lambda in default_factory (takes NO arguments)
lambda: [None, None]
#     ^
#  Nothing before the colon — zero parameters
# This is a zero-argument function that, when called,
# creates and returns a fresh [None, None] list. It's equivalent to writing:
def make_default_parents():
    return [None, None]

# These two are identical:
parents: list[Person | None] = field(default_factory=make_default_parents)
parents: list[Person | None] = field(default_factory=lambda: [None, None])

# Why does default_factory need a callable at all?
# ❌ DANGEROUS — every Person shares the SAME list object
parents: list[Person | None] = [None, None]

# If you mutated one person's parents, every person's parents would change
# because they all point to the same list in memory. This is the mutable default
# argument trap — you already know this pattern from your BMP filter work.

# default_factory solves this by requiring a callable (a function). 
# Every time a new Person is created, dataclasses calls that function
# to produce a brand-new, independent list:
#   Person A gets created → calls lambda → fresh [None, None]
#   Person B gets created → calls lambda → different fresh [None, None]
#   Person C gets created → calls lambda → another fresh [None, None]

# Each person gets their own list in memory, so mutating one person's parents
# never affects another's.

# Quick mental model
#   Think of default_factory as: "don't give me the value — give me a recipe to
#   make the value, and I'll follow that recipe each time I need a new one."



In [ ]:
# =============================================================================
# DATACLASS WITH __post_init__
# =============================================================================

from dataclasses import dataclass, field

@dataclass
class Rectangle:
    """Rectangle with computed properties."""
    width: float
    height: float
    
    # Computed after __init__
    area: float = field(init=False)
    perimeter: float = field(init=False)
    
    def __post_init__(self) -> None:
        """Called after auto-generated __init__."""
        # Validation
        if self.width <= 0 or self.height <= 0:
            raise ValueError("Dimensions must be positive")
        
        # Computed fields
        self.area = self.width * self.height
        self.perimeter = 2 * (self.width + self.height)


rect = Rectangle(10, 5)
print(rect)
# Rectangle(width=10, height=5, area=50, perimeter=30)

# Validation works
try:
    bad_rect = Rectangle(-10, 5)
except ValueError as e:
    print(f"Error: {e}")

In [ ]:
# =============================================================================
# DATACLASS DECORATOR OPTIONS
# =============================================================================

from dataclasses import dataclass

# Full signature:
# @dataclass(
#     init=True,       # Generate __init__
#     repr=True,       # Generate __repr__
#     eq=True,         # Generate __eq__
#     order=False,     # Generate __lt__, __le__, __gt__, __ge__
#     unsafe_hash=False,  # Generate __hash__
#     frozen=False,    # Make immutable
#     match_args=True, # For pattern matching (3.10+)
#     kw_only=False,   # All fields keyword-only (3.10+)
#     slots=False,     # Use __slots__ (3.10+)
# )

@dataclass(order=True)  # Enable comparison operators
class Version:
    """Sortable version numbers."""
    major: int
    minor: int
    patch: int


versions = [
    Version(2, 0, 0),
    Version(1, 5, 2),
    Version(1, 10, 0),
    Version(1, 5, 10),
]

# Now we can sort!
for v in sorted(versions):
    print(v)
# Version(major=1, minor=5, patch=2)
# Version(major=1, minor=5, patch=10)
# Version(major=1, minor=10, patch=0)
# Version(major=2, minor=0, patch=0)

In [ ]:
# =============================================================================
# DATACLASS WITH SLOTS (Python 3.10+)
# =============================================================================

from dataclasses import dataclass

@dataclass(slots=True)  # Memory efficient!
class Point:
    """Memory-efficient point."""
    x: float
    y: float


point = Point(3.0, 4.0)
print(point)  # Point(x=3.0, y=4.0)

# Cannot add new attributes
try:
    point.z = 5.0
except AttributeError as e:
    print(f"Error: {e}")

In [5]:
# =============================================================================
# DATACLASS WITH KW_ONLY (Python 3.10+)
# =============================================================================

from dataclasses import dataclass, field, KW_ONLY

@dataclass
class Request:
    """HTTP request with keyword-only fields."""
    url: str           # Positional or keyword
    method: str        # Positional or keyword
    _: KW_ONLY         # Everything after is keyword-only
    headers: dict = field(default_factory=dict)
    timeout: int = 30
    retries: int = 3


# Must use keywords for headers, timeout, retries
req = Request("https://api.com", "GET", timeout=60)
print(req)

# This would fail:
# req = Request("https://api.com", "GET", {}, 60, 3)  # TypeError

Request(url='https://api.com', method='GET', headers={}, timeout=60, retries=3)


In [ ]:
# =============================================================================
# DATACLASS INHERITANCE
# =============================================================================

from dataclasses import dataclass

@dataclass
class Animal:
    """Base animal."""
    name: str
    age: int

@dataclass
class Dog(Animal):
    """Dog extends Animal."""
    breed: str
    is_good_boy: bool = True


dog = Dog("Buddy", 3, "Labrador")
print(dog)
# Dog(name='Buddy', age=3, breed='Labrador', is_good_boy=True)

In [ ]:
# =============================================================================
# DATACLASS WITH METHODS
# =============================================================================

from dataclasses import dataclass
from typing import Self  # Python 3.11+

@dataclass
class Vector:
    """2D vector with operations."""
    x: float
    y: float
    
    def magnitude(self) -> float:
        """Calculate vector magnitude."""
        return (self.x ** 2 + self.y ** 2) ** 0.5
    
    def normalize(self) -> "Vector":
        """Return unit vector."""
        mag = self.magnitude()
        return Vector(self.x / mag, self.y / mag)
    
    def __add__(self, other: "Vector") -> "Vector":
        """Add two vectors."""
        return Vector(self.x + other.x, self.y + other.y)
    
    def __mul__(self, scalar: float) -> "Vector":
        """Scalar multiplication."""
        return Vector(self.x * scalar, self.y * scalar)


v1 = Vector(3, 4)
v2 = Vector(1, 2)

print(f"v1: {v1}")
print(f"Magnitude: {v1.magnitude()}")  # 5.0
print(f"Normalized: {v1.normalize()}")  # Vector(x=0.6, y=0.8)
print(f"v1 + v2: {v1 + v2}")  # Vector(x=4, y=6)
print(f"v1 * 2: {v1 * 2}")    # Vector(x=6, y=8)

---

## 4. Frozen Dataclasses

Immutable dataclasses that can be hashed.

In [ ]:
# =============================================================================
# FROZEN DATACLASS - Immutable
# =============================================================================

from dataclasses import dataclass

@dataclass(frozen=True)  # Cannot modify after creation
class Point:
    """Immutable point."""
    x: float
    y: float


point = Point(3, 4)
print(point)  # Point(x=3, y=4)

# Cannot modify!
try:
    point.x = 10  # FrozenInstanceError!
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

In [ ]:
# =============================================================================
# FROZEN DATACLASS AS DICT KEY / SET MEMBER
# =============================================================================

from dataclasses import dataclass

@dataclass(frozen=True)
class Coordinate:
    """Hashable coordinate."""
    x: int
    y: int


# Can use as dict key
grid: dict[Coordinate, str] = {
    Coordinate(0, 0): "origin",
    Coordinate(1, 0): "right",
    Coordinate(0, 1): "up",
}

print(grid[Coordinate(0, 0)])  # origin

# Can use in sets
visited: set[Coordinate] = {
    Coordinate(0, 0),
    Coordinate(1, 1),
}

print(Coordinate(0, 0) in visited)  # True
print(Coordinate(2, 2) in visited)  # False

In [8]:
# =============================================================================
# FROZEN DATACLASS FOR CONSTANTS
# =============================================================================

from dataclasses import dataclass
from typing import Final

@dataclass(frozen=True)
class BmpConstants:
    """BMP file format constants."""
    HEADER_SIZE: Final[int] = 14
    DIB_HEADER_SIZE: Final[int] = 40
    SIGNATURE: Final[bytes] = b"BM"
    PIXEL_SIZE: Final[int] = 3
    BPP: Final[int] = 24


# Create single instance
BMP = BmpConstants()

# Use constants
print(f"Header size: {BMP.HEADER_SIZE}")
print(f"Signature: {BMP.SIGNATURE}")

# Cannot modify
try:
    BMP.HEADER_SIZE = 100
except Exception as e:
    print(f"Error: {type(e).__name__}")

Header size: 14
Signature: b'BM'
Error: FrozenInstanceError


---

## 5. TypedDict

Type hints for dictionaries with specific keys.

In [ ]:
# =============================================================================
# BASIC TYPEDDICT
# =============================================================================

from typing import TypedDict

class User(TypedDict):
    """User data structure."""
    id: int
    name: str
    email: str


# Create (still just a dict at runtime!)
user: User = {
    "id": 1,
    "name": "Manuel",
    "email": "manuel@example.com"
}

# Access like normal dict
print(user["name"])  # Manuel
print(type(user))    # <class 'dict'>

# Type checker catches errors:
# user["invalid_key"]  # Type error
# user["id"] = "not an int"  # Type error

In [ ]:
# =============================================================================
# TYPEDDICT WITH OPTIONAL KEYS
# =============================================================================

from typing import TypedDict, Required, NotRequired

# Method 1: All optional with total=False
class ConfigOptional(TypedDict, total=False):
    """All keys are optional."""
    debug: bool
    timeout: int
    log_level: str


config1: ConfigOptional = {}  # Valid
config2: ConfigOptional = {"debug": True}  # Valid


# Method 2: Mix required and optional (Python 3.11+)
class Config(TypedDict, total=False):
    """Config with required and optional keys."""
    app_name: Required[str]       # Must be present
    api_key: Required[str]        # Must be present
    debug: NotRequired[bool]      # Optional
    timeout: NotRequired[int]     # Optional


config: Config = {
    "app_name": "MyApp",
    "api_key": "secret",
    # debug and timeout are optional
}

print(config)

In [ ]:
# =============================================================================
# TYPEDDICT INHERITANCE
# =============================================================================

from typing import TypedDict

class BaseUser(TypedDict):
    """Base user fields."""
    id: int
    name: str

class AdminUser(BaseUser):
    """Admin with extra fields."""
    permissions: list[str]
    is_superuser: bool


admin: AdminUser = {
    "id": 1,
    "name": "Admin",
    "permissions": ["read", "write", "delete"],
    "is_superuser": True
}

print(admin)

In [ ]:
# =============================================================================
# TYPEDDICT FOR API RESPONSES
# =============================================================================

from typing import TypedDict

class ApiError(TypedDict):
    """API error response."""
    code: int
    message: str

class PaginationInfo(TypedDict):
    """Pagination metadata."""
    page: int
    per_page: int
    total: int
    total_pages: int

class UserData(TypedDict):
    """User in response."""
    id: int
    name: str
    email: str

class UsersResponse(TypedDict):
    """API response for listing users."""
    success: bool
    data: list[UserData]
    pagination: PaginationInfo


# Type-safe API response handling
def process_response(response: UsersResponse) -> None:
    if response["success"]:
        for user in response["data"]:
            print(f"User: {user['name']} ({user['email']})")
        print(f"Page {response['pagination']['page']} of {response['pagination']['total_pages']}")


# Example response
response: UsersResponse = {
    "success": True,
    "data": [
        {"id": 1, "name": "Alice", "email": "alice@example.com"},
        {"id": 2, "name": "Bob", "email": "bob@example.com"},
    ],
    "pagination": {"page": 1, "per_page": 10, "total": 2, "total_pages": 1}
}

process_response(response)

In [ ]:
# =============================================================================
# TYPEDDICT AS FUNCTION RETURN TYPE
# =============================================================================

from typing import TypedDict
from pathlib import Path

class ProcessingResult(TypedDict):
    """Result from file processing."""
    input_file: str
    output_file: str
    bytes_processed: int
    duration_ms: float
    success: bool
    error_message: str | None


def process_file(input_path: Path, output_path: Path) -> ProcessingResult:
    """Process a file and return structured result."""
    # Simulate processing
    return {
        "input_file": str(input_path),
        "output_file": str(output_path),
        "bytes_processed": 1024,
        "duration_ms": 15.5,
        "success": True,
        "error_message": None,
    }


result = process_file(Path("input.txt"), Path("output.txt"))
print(f"Processed {result['bytes_processed']} bytes")
print(f"Success: {result['success']}")

---

## 6. NamedTuple

Immutable tuple with named fields.

In [ ]:
# =============================================================================
# BASIC NAMEDTUPLE
# =============================================================================

from typing import NamedTuple

class Point(NamedTuple):
    """Immutable point."""
    x: float
    y: float


point = Point(3.0, 4.0)

# Access by name
print(f"x: {point.x}, y: {point.y}")

# Access by index (it's a tuple!)
print(f"point[0]: {point[0]}")

# Unpack like tuple
x, y = point
print(f"Unpacked: {x}, {y}")

# Immutable!
try:
    point.x = 10  # AttributeError!
except AttributeError as e:
    print(f"Error: {e}")

In [ ]:
# =============================================================================
# NAMEDTUPLE WITH DEFAULTS
# =============================================================================

from typing import NamedTuple

class Config(NamedTuple):
    """Configuration with defaults."""
    host: str
    port: int = 8080
    debug: bool = False


config1 = Config("localhost")  # Uses defaults
config2 = Config("localhost", 3000, True)

print(config1)  # Config(host='localhost', port=8080, debug=False)
print(config2)  # Config(host='localhost', port=3000, debug=True)

In [ ]:
# =============================================================================
# NAMEDTUPLE WITH METHODS
# =============================================================================

from typing import NamedTuple
import math

class Vector(NamedTuple):
    """2D vector with operations."""
    x: float
    y: float
    
    def magnitude(self) -> float:
        """Calculate magnitude."""
        return math.sqrt(self.x ** 2 + self.y ** 2)
    
    def normalize(self) -> "Vector":
        """Return unit vector."""
        mag = self.magnitude()
        return Vector(self.x / mag, self.y / mag)
    
    def __add__(self, other: "Vector") -> "Vector":
        """Add vectors."""
        return Vector(self.x + other.x, self.y + other.y)


v1 = Vector(3, 4)
v2 = Vector(1, 2)

print(f"Magnitude: {v1.magnitude()}")  # 5.0
print(f"Normalized: {v1.normalize()}")  # Vector(x=0.6, y=0.8)
print(f"v1 + v2: {v1 + v2}")  # Vector(x=4, y=6)

In [ ]:
# =============================================================================
# NAMEDTUPLE _replace() METHOD
# =============================================================================

from typing import NamedTuple

class User(NamedTuple):
    name: str
    email: str
    age: int


user = User("Manuel", "manuel@example.com", 30)
print(f"Original: {user}")

# Create modified copy (original unchanged)
updated_user = user._replace(age=31)
print(f"Updated: {updated_user}")
print(f"Original still: {user}")

In [ ]:
# =============================================================================
# NAMEDTUPLE AS DICT KEY (Hashable)
# =============================================================================

from typing import NamedTuple

class Cell(NamedTuple):
    """Grid cell coordinate."""
    row: int
    col: int


# Use as dict key
grid_values: dict[Cell, str] = {
    Cell(0, 0): "A1",
    Cell(0, 1): "B1",
    Cell(1, 0): "A2",
}

print(grid_values[Cell(0, 0)])  # A1

# Use in sets
selected_cells: set[Cell] = {Cell(0, 0), Cell(1, 1), Cell(2, 2)}
print(Cell(1, 1) in selected_cells)  # True

---

## 7. Enum

Set of named constants.

In [1]:
# =============================================================================
# BASIC ENUM
# =============================================================================

from enum import Enum, auto

class Color(Enum):
    """Basic color enumeration."""
    RED = 1
    GREEN = 2
    BLUE = 3


# Access
print(Color.RED)        # Color.RED
print(Color.RED.name)   # RED
print(Color.RED.value)  # 1

# Comparison
print(Color.RED == Color.RED)  # True
print(Color.RED == Color.BLUE)  # False

# Iteration
for color in Color:
    print(f"{color.name}: {color.value}")

Color.RED
RED
1
True
False
RED: 1
GREEN: 2
BLUE: 3


In [2]:
# =============================================================================
# ENUM WITH auto()
# =============================================================================

from enum import Enum, auto

class Status(Enum):
    """Status with auto-generated values."""
    PENDING = auto()    # 1
    RUNNING = auto()    # 2
    COMPLETED = auto()  # 3
    FAILED = auto()     # 4


for status in Status:
    print(f"{status.name}: {status.value}")

PENDING: 1
RUNNING: 2
COMPLETED: 3
FAILED: 4


In [ ]:
# =============================================================================
# IntEnum - Can use as integers
# =============================================================================

from enum import IntEnum

class Priority(IntEnum):
    """Priority levels that work as integers."""
    LOW = 1
    MEDIUM = 2
    HIGH = 3
    CRITICAL = 4


# Can use in numeric operations
print(Priority.HIGH > Priority.LOW)  # True
print(Priority.HIGH + 1)  # 4

# Can compare with integers
print(Priority.MEDIUM == 2)  # True

In [ ]:
# =============================================================================
# StrEnum (Python 3.11+) - String values
# =============================================================================

from enum import StrEnum

class HttpMethod(StrEnum):
    """HTTP methods as strings."""
    GET = "GET"
    POST = "POST"
    PUT = "PUT"
    DELETE = "DELETE"


# Can use directly as string
print(f"Method: {HttpMethod.GET}")  # Method: GET
print(HttpMethod.GET == "GET")  # True

In [ ]:
# =============================================================================
# ENUM WITH METHODS
# =============================================================================

from enum import Enum

class FileType(Enum):
    """File types with extensions."""
    BMP = ".bmp"
    PNG = ".png"
    JPEG = ".jpg"
    GIF = ".gif"
    
    @classmethod
    def from_extension(cls, ext: str) -> "FileType":
        """Get FileType from extension."""
        ext = ext.lower()
        for file_type in cls:
            if file_type.value == ext:
                return file_type
        raise ValueError(f"Unknown extension: {ext}")
    
    def is_lossless(self) -> bool:
        """Check if format is lossless."""
        return self in (FileType.BMP, FileType.PNG, FileType.GIF)


# Usage
file_type = FileType.from_extension(".bmp")
print(f"{file_type.name}: lossless={file_type.is_lossless()}")
# BMP: lossless=True

In [3]:
# =============================================================================
# FLAG ENUM - Combinable values
# =============================================================================

from enum import Flag, auto

class Permission(Flag):
    """File permissions as flags."""
    READ = auto()     # 1
    WRITE = auto()    # 2
    EXECUTE = auto()  # 4
    
    # Common combinations
    RW = READ | WRITE
    RWX = READ | WRITE | EXECUTE


# Combine permissions
user_perms = Permission.READ | Permission.WRITE
print(f"Permissions: {user_perms}")  # Permission.RW

# Check permission
print(Permission.READ in user_perms)   # True
print(Permission.EXECUTE in user_perms)  # False

Permissions: Permission.RW
True
False


---

## 8. Attrs (Third-Party)

More powerful than dataclass, with validators.

In [ ]:
# =============================================================================
# ATTRS - MORE POWERFUL DATACLASS ALTERNATIVE
# =============================================================================

# Install: pip install attrs

# Note: This cell requires 'attrs' to be installed
# Uncomment and run if attrs is available

# from attrs import define, field, validators

# @define
# class User:
#     """User with validation."""
#     name: str = field(validator=validators.instance_of(str))
#     age: int = field(validator=[validators.instance_of(int), validators.ge(0)])
#     email: str = field(validator=validators.matches_re(r'.+@.+\..+'))

# # Validation happens at creation
# user = User("Manuel", 30, "manuel@example.com")  # OK
# # user = User("Manuel", -5, "invalid")  # Raises ValueError!

print("Attrs example (requires 'pip install attrs')")

---

## 9. Pydantic (Third-Party)

Runtime validation and serialization.

In [ ]:
# =============================================================================
# PYDANTIC - VALIDATION AT RUNTIME
# =============================================================================

# Install: pip install pydantic

# Note: This cell requires 'pydantic' to be installed
# Uncomment and run if pydantic is available

from pydantic import BaseModel, Field, EmailStr, field_validator

class User(BaseModel):
    """User with runtime validation."""
    id: int 
    name: str = Field(min_length=1, max_length=100)
    email: EmailStr
    age: int = Field(ge=0, le=150)
    
    @field_validator('name')
    def name_must_be_title_case(cls, v):
        return v.title()

# # Automatic type coercion
user = User(id="123", name="manuel", email="m@example.com", age="30")
print(user.id)    # 123 (converted to int)
print(user.name)  # Manuel (title case)
print(user.age)   # 30 (converted to int)

# # JSON serialization
print(user.model_dump_json())

print("Pydantic example (requires 'pip install pydantic')")

123
Manuel
30
{"id":123,"name":"Manuel","email":"m@example.com","age":30}
Pydantic example (requires 'pip install pydantic')


---

## 10. Comparison & Best Practices

In [4]:
# =============================================================================
# SIDE-BY-SIDE COMPARISON
# =============================================================================

from dataclasses import dataclass
from typing import TypedDict, NamedTuple

# Same data structure in different forms:

# 1. Dataclass (most common)
@dataclass
class UserDataclass:
    name: str
    age: int

# 2. Frozen dataclass (immutable)
@dataclass(frozen=True)
class UserFrozen:
    name: str
    age: int

# 3. TypedDict (dict with type hints)
class UserTypedDict(TypedDict):
    name: str
    age: int

# 4. NamedTuple (immutable, tuple-like)
class UserNamedTuple(NamedTuple):
    name: str
    age: int


# Create instances
dc = UserDataclass("Manuel", 30)
frozen = UserFrozen("Manuel", 30)
td: UserTypedDict = {"name": "Manuel", "age": 30}
nt = UserNamedTuple("Manuel", 30)

print(f"Dataclass: {dc}, type={type(dc).__name__}")
print(f"Frozen: {frozen}, type={type(frozen).__name__}")
print(f"TypedDict: {td}, type={type(td).__name__}")
print(f"NamedTuple: {nt}, type={type(nt).__name__}")

Dataclass: UserDataclass(name='Manuel', age=30), type=UserDataclass
Frozen: UserFrozen(name='Manuel', age=30), type=UserFrozen
TypedDict: {'name': 'Manuel', 'age': 30}, type=dict
NamedTuple: UserNamedTuple(name='Manuel', age=30), type=UserNamedTuple


In [ ]:
# =============================================================================
# FEATURE COMPARISON
# =============================================================================

print("""
╔════════════════════╦══════════╦══════════════╦═══════════╦════════════╗
║ Feature            ║ dataclass║ frozen DC    ║ TypedDict ║ NamedTuple ║
╠════════════════════╬══════════╬══════════════╬═══════════╬════════════╣
║ Mutable            ║ ✓        ║ ✗            ║ ✓         ║ ✗          ║
║ Hashable           ║ ✗*       ║ ✓            ║ ✗         ║ ✓          ║
║ Dict key           ║ ✗*       ║ ✓            ║ ✗         ║ ✓          ║
║ Attribute access   ║ obj.attr ║ obj.attr     ║ d["key"]  ║ obj.attr   ║
║ Index access       ║ ✗        ║ ✗            ║ ✓         ║ ✓          ║
║ JSON serializable  ║ ✗        ║ ✗            ║ ✓         ║ ✗          ║
║ Methods            ║ ✓        ║ ✓            ║ ✗         ║ ✓          ║
║ Inheritance        ║ ✓        ║ ✓            ║ ✓         ║ ✓          ║
║ Default values     ║ ✓        ║ ✓            ║ partial   ║ ✓          ║
║ Type checking only ║ ✗        ║ ✗            ║ ✓         ║ ✗          ║
║ Runtime type       ║ class    ║ class        ║ dict      ║ tuple      ║
╚════════════════════╩══════════╩══════════════╩═══════════╩════════════╝

* Can be made hashable with unsafe_hash=True
""")

In [ ]:
# =============================================================================
# WHEN TO USE WHAT
# =============================================================================

print("""
DECISION GUIDE:

Use DATACLASS when:
  • You need a simple data container
  • You want auto-generated __init__, __repr__, __eq__
  • You might add methods later
  • You need mutable objects

Use FROZEN DATACLASS when:
  • You need immutability
  • You want to use objects as dict keys or in sets
  • You're creating value objects (like coordinates, versions)

Use TYPEDDICT when:
  • You're working with JSON/API responses
  • You need dict operations (.get(), .items(), etc.)
  • You want type hints for existing dict structures
  • You need to serialize to JSON easily

Use NAMEDTUPLE when:
  • You need immutability AND tuple operations
  • You want to unpack values (x, y = point)
  • Memory efficiency matters (smaller than dataclass)
  • You're replacing tuples with named access

Use REGULAR CLASS when:
  • You need complex initialization logic
  • You need encapsulation (private attributes)
  • You need properties with getters/setters
  • You have significant behavior (methods)

Use ENUM when:
  • You have a fixed set of choices
  • You want type-safe constants
  • You need to iterate over options

Use PYDANTIC when:
  • You need runtime validation
  • You're building APIs (FastAPI integration)
  • You need automatic type coercion
  • You need JSON Schema generation
""")

---

## 11. Real-World Patterns

In [ ]:
# =============================================================================
# PATTERN: Configuration with Dataclass
# =============================================================================

from dataclasses import dataclass, field
from pathlib import Path

@dataclass
class DatabaseConfig:
    """Database configuration."""
    host: str
    port: int
    database: str
    username: str
    password: str = field(repr=False)  # Don't show in repr
    
    # @property lets you access a method as if it were an attribute — no
    # parentheses needed. It looks like reading a variable but runs a
    # function behind the scenes.
    @property
    def connection_string(self) -> str:
        return f"postgresql://{self.username}:{self.password}@{self.host}:{self.port}/{self.database}"

@dataclass
class AppConfig:
    """Application configuration."""
    app_name: str
    debug: bool = False
    log_level: str = "INFO"
    database: DatabaseConfig = field(default_factory=lambda: DatabaseConfig(
        host="localhost",
        port=5432,
        database="app",
        username="user",
        password="secret"
    ))


config = AppConfig("MyApp", debug=True)
print(config)
print(f"Connection: {config.database.connection_string}")

# =============================================================================

# Three levels: getter, setter, deleter
class ImageConfig:
    def __init__(self, width: int, height: int) -> None:
        self._width = width       # underscore = "private, use the property"
        self._height = height

    # ── GETTER: @property ──────────────────────────────────
    @property
    def width(self) -> int:
        """Image width in pixels."""
        return self._width

    # ── SETTER: @name.setter ──────────────────────────────
    @width.setter
    def width(self, value: int) -> None:
        if value <= 0:
            raise ValueError("Width must be positive")
        self._width = value

    # ── DELETER: @name.deleter (rare) ─────────────────────
    @width.deleter
    def width(self) -> None:
        raise AttributeError("Cannot delete width")
    
    # All three use the same name (width). Python knows which to call based on 
    # the operation:
cfg = ImageConfig(640, 480)

cfg.width             # calls the GETTER → 640
cfg.width = 1920      # calls the SETTER → validates, then stores
cfg.width = -1        # calls the SETTER → raises ValueError!
del cfg.width         # calls the DELETER → raises AttributeError!

# Why the underscore convention self._width
# When you use @property with a setter, you need a place to actually store the value.
# If the property is called width and the stored attribute is also called width,
# you get infinite recursion:
# BAD — infinite recursion
@property
def width(self) -> int:
    return self.width        # calls the property getter again → loops forever

@width.setter
def width(self, value: int) -> None:
    self.width = value       # calls the property setter again → loops forever
    
# The convention is to store in self._width (private) and expose through the self.width
# property (public). 
# The underscore signals "access this through the property, not directly."

# Use @property when it READS like an attribute:
img.width              # "what is the width?" — a characteristic
img.total_pixels       # "how many pixels?" — a derived fact
img.is_valid           # "is it valid?" — a state check

# Use a regular method when it DOES something:
img.apply_filter()     # performs an action
img.save("out.bmp")    # side effect (writes to disk)
img.calculate_histogram()  # expensive computation (signal with ())

# The parentheses serve as a signal to the caller: "this might be slow or have
# side effects." Properties should be cheap — if computing the value takes noticeable
# time, use a method so the caller knows there's a cost.

In [ ]:
# =============================================================================
# PATTERN: Value Objects with Frozen Dataclass
# =============================================================================

from dataclasses import dataclass
from typing import Self

@dataclass(frozen=True, order=True)
class Money:
    """Immutable money value object."""
    amount: int  # Store as cents to avoid float issues
    currency: str = "USD"
    
    def __post_init__(self) -> None:
        if self.amount < 0:
            raise ValueError("Amount cannot be negative")
    
    # When you call Money.from_dollars(15.00), Python sets cls = Money. Then
    # cls(int(dollars * 100), currency) is the same as Money(1500, "USD")
    # Self means "returns an instance of whatever class this was called on".
    @classmethod    # "this method gets the CLASS, not an instance"
    def from_dollars(cls, dollars: float, currency: str = "USD") -> Self:
        """Create from dollar amount."""
        return cls(int(dollars * 100), currency)
        #      ^^^ This calls cls() which is Money() = the class constructor.
        # We use cls instead of Money directly, because of inheritance. If
        # another developer uses Money to inherit from in their own class,
        # cls would be their own instance, instead of a 'Money' instance.
    
    def to_dollars(self) -> float:
        """Convert to dollar amount."""
        return self.amount / 100
    
    def __add__(self, other: "Money") -> "Money":
        if self.currency != other.currency:
            raise ValueError("Cannot add different currencies")
        return Money(self.amount + other.amount, self.currency)
    
    # Implement dunder __str__ to be triggered by default by an f-string,
    # f" {..}" inside print() or logger.debug() or 'return f".."'.
    def __str__(self) -> str:
        return f"${self.to_dollars():.2f} {self.currency}"


price1 = Money.from_dollars(19.99)
price2 = Money.from_dollars(5.99)
total = price1 + price2

print(f"Price 1: {price1}")
print(f"Price 2: {price2}")
print(f"Total: {total}")

Price 1: $19.98 USD
Price 2: $5.99 USD
Total: $25.97 USD


In [ ]:
# =============================================================================
# PATTERN: API Response with TypedDict
# =============================================================================

from typing import TypedDict, Required, NotRequired
import json

class ErrorDetail(TypedDict):
    field: str
    message: str

class ApiResponse(TypedDict, total=False):
    """Generic API response structure."""
    success: Required[bool]
    data: NotRequired[dict]
    errors: NotRequired[list[ErrorDetail]]
    message: NotRequired[str]


def create_success_response(data: dict) -> ApiResponse:
    """Create success response."""
    return {
        "success": True,
        "data": data,
    }

def create_error_response(errors: list[ErrorDetail]) -> ApiResponse:
    """Create error response."""
    return {
        "success": False,
        "errors": errors,
    }


# Usage
success = create_success_response({"id": 1, "name": "Manuel"})
error = create_error_response([{"field": "email", "message": "Invalid email"}])

# Easy JSON serialization
print(json.dumps(success, indent=2))
print(json.dumps(error, indent=2))

In [7]:
# =============================================================================
# PATTERN: State Machine with Enum
# =============================================================================

from enum import Enum, auto
from dataclasses import dataclass, field
from datetime import datetime

class OrderStatus(Enum):
    """Order status states."""
    PENDING = auto()
    CONFIRMED = auto()
    PROCESSING = auto()
    SHIPPED = auto()
    DELIVERED = auto()
    CANCELLED = auto()
    
    def can_transition_to(self, new_status: "OrderStatus") -> bool:
        """Check if transition is valid."""
        valid_transitions = {
            OrderStatus.PENDING: {OrderStatus.CONFIRMED, OrderStatus.CANCELLED},
            OrderStatus.CONFIRMED: {OrderStatus.PROCESSING, OrderStatus.CANCELLED},
            OrderStatus.PROCESSING: {OrderStatus.SHIPPED, OrderStatus.CANCELLED},
            OrderStatus.SHIPPED: {OrderStatus.DELIVERED},
            OrderStatus.DELIVERED: set(),
            OrderStatus.CANCELLED: set(),
        }
        return new_status in valid_transitions.get(self, set())


@dataclass
class Order:
    """Order with status management."""
    id: int
    customer: str
    status: OrderStatus = OrderStatus.PENDING
    history: list[tuple[OrderStatus, datetime]] = field(default_factory=list)
    
    def __post_init__(self):
        self.history.append((self.status, datetime.now()))
    
    def transition_to(self, new_status: OrderStatus) -> bool:
        """Attempt status transition."""
        if self.status.can_transition_to(new_status):
            self.status = new_status
            self.history.append((new_status, datetime.now()))
            return True
        return False


# Usage
order = Order(1, "Manuel")
print(f"Initial: {order.status.name}")

order.transition_to(OrderStatus.CONFIRMED)
print(f"After confirm: {order.status.name}")

order.transition_to(OrderStatus.PROCESSING)
print(f"After processing: {order.status.name}")

# Invalid transition
result = order.transition_to(OrderStatus.PENDING)
print(f"Back to pending: {result}")  # False

Initial: PENDING
After confirm: CONFIRMED
After processing: PROCESSING
Back to pending: False


---

## 📋 Quick Reference

```python
# =============================================================================
# DATA STRUCTURES QUICK REFERENCE
# =============================================================================
#
# DATACLASS:
#     @dataclass
#     class User:
#         name: str
#         age: int = 0
#         tags: list = field(default_factory=list)
#
# FROZEN DATACLASS:
#     @dataclass(frozen=True)  # Immutable, hashable
#     class Point:
#         x: float
#         y: float
#
# TYPEDDICT:
#     class User(TypedDict):
#         name: str
#         age: int
#     user: User = {"name": "X", "age": 30}
#
# NAMEDTUPLE:
#     class Point(NamedTuple):
#         x: float
#         y: float
#     p = Point(1, 2)
#     x, y = p  # Unpack
#
# ENUM:
#     class Color(Enum):
#         RED = 1
#         GREEN = 2
#
# INTENUM (for comparisons):
#     class Priority(IntEnum):
#         LOW = 1
#         HIGH = 2
#
# STRENUM (Python 3.11+):
#     class Status(StrEnum):
#         ACTIVE = "active"
#
# =============================================================================
```